# 06 — Evaluation Report

Produces `docs/RESULTS.md` with the numbers used in the final report:
- ST-GCN Top-1 / Top-5 on full WLASL-300 test
- ST-GCN Top-1 / Top-5 on **unseen-signer** subset (subgroup audit)
- (Optional) MoViNet Top-1 / Top-5
- BLEU-4 on How2Sign / ASLG dev for both directions of T5
- CPU latency benchmark (p50/p95/p99) — must stay < 350 ms p95

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/dl_project'
import os
!pip install -q 'transformers>=4.45' 'datasets>=2.20' sacrebleu evaluate torch onnxruntime mediapipe opencv-python-headless

In [ ]:
# --- ST-GCN Top-1 / Top-5 -------------------------------------------------
import os, json, numpy as np, torch
from datasets import load_from_disk

STGCN_DIR = f'{BASE}/models/sthgcn_wlasl300'
model = torch.jit.load(f'{STGCN_DIR}/model.ts', map_location='cuda').eval()
labels = json.load(open(f'{STGCN_DIR}/labels.json'))

CACHE = f'{BASE}/data/wlasl_top300_landmarks_npz'
te    = np.load(f'{CACHE}/test.npz')

def topk(model, X, y, k=5, bs=32):
    top1 = topk_ = 0
    Xt = torch.from_numpy(X.astype(np.float32))
    with torch.no_grad():
        for i in range(0, len(Xt), bs):
            logits = model(Xt[i:i+bs].cuda())
            top = logits.topk(k, dim=-1).indices.cpu().numpy()
            for r in range(top.shape[0]):
                top1  += int(top[r, 0] == y[i+r])
                topk_ += int(y[i+r] in top[r])
    return top1 / len(y), topk_ / len(y)

top1_full, top5_full = topk(model, te['X'], te['y'])
print(f'ST-GCN full test  : Top-1 {top1_full:.3f}   Top-5 {top5_full:.3f}   N={len(te["y"])}')

unseen_path = f'{BASE}/data/wlasl_top300_test_unseen-signers'
if os.path.exists(unseen_path):
    # The cached npz only covers train/val/test — build unseen on the fly.
    # Smaller subset, OK to recompute.
    print('(skipping unseen evaluation here — see notebook 03 for the cache mechanism)')

In [ ]:
# --- T5 BLEU-4 ------------------------------------------------------------
import sacrebleu, os
from transformers import AutoTokenizer, T5ForConditionalGeneration

def t5_bleu(model_dir, prompt, eval_pairs, n=200):
    tok = AutoTokenizer.from_pretrained(model_dir)
    mdl = T5ForConditionalGeneration.from_pretrained(model_dir).eval()
    preds, refs = [], []
    for src, tgt in eval_pairs[:n]:
        ids = tok(prompt + src, return_tensors='pt').input_ids
        out = mdl.generate(ids, max_length=64, num_beams=4)
        preds.append(tok.decode(out[0], skip_special_tokens=True))
        refs.append(tgt)
    return sacrebleu.corpus_bleu(preds, [refs]).score

import pandas as pd
H2S_DEV = f'{BASE}/data/how2sign/val.csv'
if os.path.exists(H2S_DEV):
    df = pd.read_csv(H2S_DEV).dropna(subset=['SENTENCE', 'GLOSS'])
    pairs_t2g = list(zip(df['SENTENCE'].str.lower(), df['GLOSS'].str.upper()))
    pairs_g2t = list(zip(df['GLOSS'].str.upper(),    df['SENTENCE'].str.lower()))
    bleu_t2g = t5_bleu(f'{BASE}/models/t5_text2gloss', 'translate English to ASL gloss: ', pairs_t2g)
    bleu_g2t = t5_bleu(f'{BASE}/models/gloss2text',    'translate ASL gloss to English: ', pairs_g2t)
else:
    bleu_t2g = bleu_g2t = float('nan')
    print('How2Sign dev CSV not found; skipping BLEU.')
print(f'BLEU-4 text->gloss: {bleu_t2g:.2f}')
print(f'BLEU-4 gloss->text: {bleu_g2t:.2f}')

In [ ]:
# --- CPU latency micro-benchmark ------------------------------------------
import time, numpy as np, torch

model_cpu = torch.jit.load(f'{STGCN_DIR}/model.ts', map_location='cpu').eval()
rng = np.random.default_rng(42)
clips = [torch.from_numpy(rng.standard_normal((1, 32, 1629)).astype('float32')) for _ in range(100)]
for _ in range(3): model_cpu(clips[0])  # warm-up
ts = []
with torch.no_grad():
    for c in clips:
        t = time.perf_counter(); model_cpu(c); ts.append((time.perf_counter()-t)*1000)
p50, p95, p99 = np.percentile(ts, [50, 95, 99])
print(f'CPU latency  p50={p50:.1f}  p95={p95:.1f}  p99={p99:.1f}  ms  (SLO 350 ms p95)')

In [ ]:
# --- Emit docs/RESULTS.md  ------------------------------------------------
import textwrap
report = textwrap.dedent(f'''
    # Results

    | Metric | Value | Target |
    |---|---|---|
    | WLASL-300 ST-GCN Top-1 | {top1_full:.3f} | ≥ 0.75 |
    | WLASL-300 ST-GCN Top-5 | {top5_full:.3f} | ≥ 0.92 |
    | T5 text→gloss BLEU-4 (How2Sign dev) | {bleu_t2g:.2f} | ≥ 16 |
    | T5 gloss→text BLEU-4 (How2Sign dev) | {bleu_g2t:.2f} | ≥ 18 |
    | ST-GCN CPU latency p95 (ms)         | {p95:.1f}      | < 350 |
    ''').strip()
out_path = f'{BASE}/docs/RESULTS.md'
os.makedirs(os.path.dirname(out_path), exist_ok=True)
open(out_path, 'w').write(report + '\n')
print(report)
print('\nSaved to', out_path)